# Certified Neural Lyapunov Functions, minimal Colab

Runs the core certification end to end on a free CPU Colab: reproduce a short
Algorithm-1 training, then use CROWN with branch-and-bound to check the Lyapunov
conditions over a region, first the positive-definiteness condition (4a) and then
the gen-5 rung of the Lie-derivative condition (4b), where the as-trained network
fails with a genuine counterexample and the counterexample-guided network
certifies the annulus.

Sehaj Singh, with Prof. Wenqi Cui (NYU Tandon ECE), 2026. Simulation only.
Full runs, all seeds, all rungs, and the paper are in the repo via `make full`
and `make paper`.

In [ ]:
# 1. dependencies. torch CPU wheel, then auto_LiRPA from GitHub main (the PyPI
#    wheel lacks the sin/cos and Jacobian support this needs).
!pip -q install torch --index-url https://download.pytorch.org/whl/cpu
!pip -q install numpy scipy matplotlib
!pip -q install --ignore-requires-python "git+https://github.com/Verified-Intelligence/auto_LiRPA.git"
import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

In [ ]:
# 2. get the code and data (the .mat files ship in the repo under upstream/).
!git clone -q https://github.com/sehajr-singhs/certified-neural-lyapunov.git
%cd certified-neural-lyapunov

In [ ]:
# 3. reproduce a short Algorithm-1 training (E0) and close the loop (E4 CEGIS).
#    A few minutes on CPU. The full runs use more iterations.
!python experiments/e0_reproduce.py --seed 0 --iters 1500
!python experiments/e4_cegis.py --seed 0 --iters 3

In [ ]:
# 4. certify condition (4a): as-trained violates, CEGIS certifies.
!python experiments/e2_certify_4a.py --seed 0

In [ ]:
# 5. certify the Lie-derivative condition (4b) on the gen-5 slice, and the
#    two-machine correctness gate.
!python experiments/e3_certify_lie.py --seed 0
!python experiments/e3_rung1_twobus.py --seed 0

In [ ]:
# 6. the headline numbers.
import json
e3 = json.load(open("results/e3_seed0.json"))
print("as-trained (4b) gen-5 far region:", e3["rung2"]["as_trained_far_region"]["verdict"],
      "Lie =", e3["rung2"]["as_trained_far_region"]["lie_at_ce"])
print("CEGIS (4b) annulus certified rho:", e3["rung2"]["cegis_4b_annulus"]["certified_rho"])
print("CEGIS Prop-2 certified beta:", e3["rung2"]["cegis_prop2_beta_max"])

## dReal SMT baseline

The comparison the paper is measured against. SMT-based neural Lyapunov verification
(Chang, Roohi, Gao, NeurIPS 2019) is Cui's reference [14], and her paper dismisses it
as tractable only for small systems, so this is the argument her citation sets up.
dReal only runs on Linux, which is why it is here in Colab and not in the Windows
repo. Rung 1 (the two-bus gate) should close, rung 2 (the NE39 slice) is expected to
exceed its budget, which is the sentence the related-work section wants.

In [ ]:
!pip -q install dreal
!python experiments/dreal_baseline.py --rung 1 --budget 600
!python experiments/dreal_baseline.py --rung 2 --budget 600